# IOAI — 2026 Local Roai Soil Quality (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-soil.zip', '/tmp/soil.zip')
    with zipfile.ZipFile('/tmp/soil.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Soil Quality (Calitatea solului) — 토양 적합성 (RoAI 2026 Local)

토양 표 데이터로 5 서브태스크:
1. (결정적) Nutrient_Index = 0.4*N + 0.3*P + 0.3*K
2. (결정적) pH 분류 Acid/Neutral/Alkaline
3. (결정적) Moisture > train중앙값 여부
4. (결정적) Soil_Type 빈도 매핑
5. (ML) `Suitability` 분류 — F1(pos=Unfavorable) 로 채점

**제출**: `submission.csv` 롱포맷 `subtaskID, datapointID, answer`.
**채점(로컬)**: ST1~4 exact, ST5 train held-out F1. **데이터**: `data/train_data.csv`(+`data/test_data.csv`).
원본: judge.nitro-ai.org/competitions (ROAI local)

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

root_path = "data"
train_df = pd.read_csv(f"{root_path}/train_data.csv")
test_df = pd.read_csv(f"{root_path}/test_data.csv")
train_df.head()

In [ ]:
# ST1~4 (결정적 규칙)
def cls_ph(ph):
    return "Acid" if ph < 6.0 else ("Neutral" if ph <= 7.5 else "Alkaline")
subtask1 = 0.4*test_df["Nitrogen"] + 0.3*test_df["Phosphorus"] + 0.3*test_df["Potassium"]  # Nutrient_Index
subtask2 = test_df["pH"].apply(cls_ph)
subtask3 = (test_df["Moisture"] > train_df["Moisture"].median()).astype(int)
train_counts = train_df["Soil_Type"].value_counts()
subtask4 = test_df["Soil_Type"].map(train_counts)

In [ ]:
# ST5 (ML, 채점 대상): 적합성 분류 — 간단 베이스라인(수치형 표준화 LogReg).
# 모범답안은 pH클래스·영양지수·범주 원핫 등 feature 엔지니어링으로 F1 을 올립니다.
feat = [c for c in train_df.select_dtypes(include=np.number).columns if c not in ("ID",)]
mean = train_df[feat].mean()
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf.fit(train_df[feat].fillna(mean), train_df["Suitability"])
subtask5 = clf.predict(test_df[feat].fillna(mean))

In [ ]:
# 제출(롱포맷): subtaskID, datapointID, answer
def build_subtask(sid, ans):
    return pd.DataFrame({"subtaskID": sid, "datapointID": test_df["ID"], "answer": np.asarray(ans)})
submission_df = pd.concat([build_subtask(i, a) for i, a in [(1,subtask1),(2,subtask2),(3,subtask3),(4,subtask4),(5,subtask5)]])
submission_df.to_csv(f"{root_path}/submission.csv", index=False)
submission_df.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)